In [0]:
%run "../utils/00_config"

In [0]:
# IMPORTS AND CONFIGURATION
from datetime import datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType,
    LongType
)
from delta.tables import DeltaTable


# TARGET TABLES
TABLE_RAW_RESPONSES = get_table("bronze", "raw_api_responses")
TABLE_ITEM_EXECUTION = get_table("bronze", "item_execution")
TABLE_INGESTION_LOG = get_table("bronze", "ingestion_log")

REQUIRED_FILES = {
    "manifest.json",
    "raw_api_responses.jsonl.gz",
    "item_execution.jsonl.gz"
}

RUNNING_TIMEOUT_HOURS = 2

print(f"Environment: {ENV.upper()}")
print(f"Raw path: {RAW_PATH}")



In [0]:
# INGESTION LOG INITIALIZATION
ingestion_log_schema = StructType([
    StructField("execution_id", StringType(), False),
    StructField("execution_date", StringType(), True),
    StructField("source_status", StringType(), True),
    StructField("ingestion_status", StringType(), False),
    StructField("ingestion_started_at", TimestampType(), True),
    StructField("ingestion_finished_at", TimestampType(), True),
    StructField("raw_response_count", LongType(), True),
    StructField("item_count", LongType(), True),
    StructField("error_message", StringType(), True),
    StructField("updated_at", TimestampType(), True)
])

if not spark.catalog.tableExists(TABLE_INGESTION_LOG):

    (
        spark.createDataFrame([], ingestion_log_schema)
        .write
        .format("delta")
        .saveAsTable(TABLE_INGESTION_LOG)
    )

    print(f"Created ingestion log: {TABLE_INGESTION_LOG}")

else:
    print(f"Ingestion log available: {TABLE_INGESTION_LOG}")

In [0]:

# INGESTION LOG UPDATE FUNCTION
def update_ingestion_log(
    execution_id,
    execution_date,
    source_status,
    ingestion_status,
    started_at=None,
    response_count=None,
    item_count=None,
    error_message=None
):

    now = datetime.now(timezone.utc).replace(tzinfo=None)

    finished_at = (
        now if ingestion_status in {"SUCCESS", "FAILED"} else None
    )

    log_data = [(
        execution_id,
        execution_date,
        source_status,
        ingestion_status,
        started_at,
        finished_at,
        response_count,
        item_count,
        error_message[:4000] if error_message else None,
        now
    )]

    log_df = spark.createDataFrame(
        log_data,
        ingestion_log_schema
    )

    target = DeltaTable.forName(spark, TABLE_INGESTION_LOG)

    (
        target.alias("target")
        .merge(
            log_df.alias("source"),
            "target.execution_id = source.execution_id"
        )
        .whenMatchedUpdate(set={
            "execution_date": "source.execution_date",
            "source_status": "source.source_status",
            "ingestion_status": "source.ingestion_status",
            "ingestion_started_at": """
                COALESCE(source.ingestion_started_at, target.ingestion_started_at)
            """,
            "ingestion_finished_at": "source.ingestion_finished_at",
            "raw_response_count": """
                COALESCE(source.raw_response_count, target.raw_response_count)
            """,
            "item_count": """
                COALESCE(source.item_count, target.item_count)
            """,
            "error_message": "source.error_message",
            "updated_at": "source.updated_at"
        })
        .whenNotMatchedInsertAll()
        .execute()
    )


In [0]:
# EXECUTION DISCOVERY
executions = []

for date_dir in dbutils.fs.ls(RAW_PATH):

    if not date_dir.name.startswith("execution_date="):
        continue

    execution_date = date_dir.name.rstrip("/").split("=", 1)[1]

    for execution_dir in dbutils.fs.ls(date_dir.path):

        if not execution_dir.name.startswith("execution_id="):
            continue

        execution_id = execution_dir.name.rstrip("/").split("=", 1)[1]

        executions.append({
            "execution_date": execution_date,
            "execution_id": execution_id,
            "raw_path": execution_dir.path
        })


executions = sorted(
    executions,
    key=lambda x: (x["execution_date"], x["execution_id"])
)

print(f"Discovered executions: {len(executions)}")

In [0]:

# EXECUTION FILTERING
log_df = spark.table(TABLE_INGESTION_LOG)

# Identify executions currently present in the raw storage.
discovered_ids = {
    execution["execution_id"]
    for execution in executions
}

# Previously completed executions.
completed_ids = {
    row["execution_id"]
    for row in (
        log_df
        .filter(F.col("ingestion_status") == "SUCCESS")
        .select("execution_id")
        .collect()
    )
} & discovered_ids

# RUNNING executions within the allowed timeout.
recent_running_ids = {
    row["execution_id"]
    for row in (
        log_df
        .filter(
            (F.col("ingestion_status") == "RUNNING")
            & (
                F.col("updated_at")
                >= F.expr(
                    f"current_timestamp() - INTERVAL {RUNNING_TIMEOUT_HOURS} HOURS"
                )
            )
        )
        .select("execution_id")
        .collect()
    )
} & discovered_ids

# RUNNING executions that exceeded the timeout.
all_running_ids = {
    row["execution_id"]
    for row in (
        log_df
        .filter(F.col("ingestion_status") == "RUNNING")
        .select("execution_id")
        .collect()
    )
} & discovered_ids

stale_running_ids = all_running_ids - recent_running_ids

# Previously failed executions eligible for retry.
failed_ids = {
    row["execution_id"]
    for row in (
        log_df
        .filter(F.col("ingestion_status") == "FAILED")
        .select("execution_id")
        .collect()
    )
} & discovered_ids

# Executions without an ingestion log record.
logged_ids = {
    row["execution_id"]
    for row in (
        log_df
        .select("execution_id")
        .collect()
    )
}

new_ids = discovered_ids - logged_ids

# Process new, failed, and stale RUNNING executions.
pending_executions = [
    execution
    for execution in executions
    if execution["execution_id"] not in completed_ids
    and execution["execution_id"] not in recent_running_ids
]

print(f"Discovered: {len(discovered_ids)}")
print(f"Already completed: {len(completed_ids)}")
print(f"Currently running: {len(recent_running_ids)}")
print(f"Stale RUNNING (eligible for retry): {len(stale_running_ids)}")
print(f"Previously failed (eligible for retry): {len(failed_ids)}")
print(f"New executions: {len(new_ids)}")
print(f"Pending for processing: {len(pending_executions)}")


In [0]:
# VALIDATION FUNCTIONS
def validate_unique_key(df, columns, table_name):

    null_keys = (
        df.filter(
            F.expr(
                " OR ".join(f"`{column}` IS NULL" for column in columns)
            )
        )
        .limit(1)
        .count()
    )

    if null_keys:
        raise ValueError(
            f"Null values found in key columns of {table_name}: {columns}"
        )

    duplicates = (
        df.groupBy(*columns)
        .count()
        .filter(F.col("count") > 1)
        .limit(1)
        .count()
    )

    if duplicates:
        raise ValueError(
            f"Duplicate key found in {table_name}: {columns}"
        )


def validate_manifest(manifest, execution_id, raw_responses, item_execution):

    required_fields = {
        "execution_id",
        "started_at",
        "finished_at",
        "status",
        "response_count",
        "item_count",
        "order_response_count",
        "total_orders"
    }

    missing_fields = required_fields - manifest.keys()

    if missing_fields:
        raise ValueError(
            f"Missing manifest fields: {missing_fields}"
        )

    if manifest["execution_id"] != execution_id:
        raise ValueError("Manifest execution_id mismatch.")

    if manifest["status"] != "success":
        raise ValueError(
            f"Collection status is {manifest['status']}."
        )

    if not manifest["started_at"] or not manifest["finished_at"]:
        raise ValueError("Missing execution timestamps.")

    if manifest["response_count"] != raw_responses.count():
        raise ValueError("Manifest response count mismatch.")

    if manifest["item_count"] != item_execution.count():
        raise ValueError("Manifest item count mismatch.")

In [0]:
# BRONZE MERGE FUNCTION
def merge_bronze(df, table_name, keys):

    if not spark.catalog.tableExists(table_name):

        (
            df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(table_name)
        )

        return

    target = DeltaTable.forName(spark, table_name)

    merge_condition = " AND ".join(
        [f"target.`{key}` = source.`{key}`" for key in keys]
    )

    (
        target.alias("target")
        .merge(df.alias("source"), merge_condition)
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

In [0]:
# EXECUTION PROCESSING
successful_executions = 0
failed_executions = []


for execution in pending_executions:

    execution_id = execution["execution_id"]
    execution_date = execution["execution_date"]
    execution_raw_path = execution["raw_path"]

    started_at = datetime.now(timezone.utc).replace(tzinfo=None)
    source_status = None

    print(f"\nProcessing execution: {execution_id}")

    try:

        # ----------------------------------
        # FILE VERIFICATION
        # ----------------------------------

        files = {
            file.name.rstrip("/")
            for file in dbutils.fs.ls(execution_raw_path)
        }

        missing_files = REQUIRED_FILES - files

        if missing_files:
            raise FileNotFoundError(
                f"Missing source files: {missing_files}"
            )

        manifest_path = execution_raw_path + "manifest.json"
        responses_path = execution_raw_path + "raw_api_responses.jsonl.gz"
        item_execution_path = execution_raw_path + "item_execution.jsonl.gz"

        # ----------------------------------
        # DATA LOADING
        # ----------------------------------

        manifest_rows = (
            spark.read
            .option("multiLine", True)
            .json(manifest_path)
            .collect()
        )

        if len(manifest_rows) != 1:
            raise ValueError(
                "Expected exactly one manifest record."
            )

        manifest = manifest_rows[0].asDict(recursive=True)
        source_status = manifest.get("status")

        raw_responses = spark.read.json(responses_path)
        item_execution = spark.read.json(item_execution_path)

        # ----------------------------------
        # MANIFEST VALIDATION
        # ----------------------------------

        validate_manifest(
            manifest,
            execution_id,
            raw_responses,
            item_execution
        )

        # ----------------------------------
        # DATAFRAME VALIDATION
        # ----------------------------------

        for name, df in {
            "raw_api_responses": raw_responses,
            "item_execution": item_execution
        }.items():

            invalid_execution = df.filter(
                F.col("execution_id").isNull()
                | (F.col("execution_id") != execution_id)
            ).limit(1).count()

            if invalid_execution:
                raise ValueError(
                    f"{name}: invalid execution_id."
                )

        validate_unique_key(
            raw_responses,
            ["execution_id", "event_type", "resource_slug"],
            "raw_api_responses"
        )

        validate_unique_key(
            item_execution,
            ["execution_id", "item_slug"],
            "item_execution"
        )

        response_count = raw_responses.count()
        item_count = item_execution.count()

        # ----------------------------------
        # MARK EXECUTION AS RUNNING
        # ----------------------------------

        update_ingestion_log(
            execution_id=execution_id,
            execution_date=execution_date,
            source_status=source_status,
            ingestion_status="RUNNING",
            started_at=started_at
        )

        # ----------------------------------
        # BRONZE DELTA INGESTION
        # ----------------------------------

        merge_bronze(
            raw_responses,
            TABLE_RAW_RESPONSES,
            ["execution_id", "event_type", "resource_slug"]
        )

        merge_bronze(
            item_execution,
            TABLE_ITEM_EXECUTION,
            ["execution_id", "item_slug"]
        )

        # ----------------------------------
        # POST-INGESTION VALIDATION
        # ----------------------------------

        validations = [
            (
                raw_responses,
                TABLE_RAW_RESPONSES,
                "raw_api_responses",
                ["execution_id", "event_type", "resource_slug"]
            ),
            (
                item_execution,
                TABLE_ITEM_EXECUTION,
                "item_execution",
                ["execution_id", "item_slug"]
            )
        ]

        for source_df, table_name, label, keys in validations:

            expected_count = source_df.count()

            persisted_df = (
                spark.table(table_name)
                .filter(F.col("execution_id") == execution_id)
            )

            persisted_count = persisted_df.count()

            if expected_count != persisted_count:
                raise ValueError(
                    f"{label}: expected {expected_count}, "
                    f"found {persisted_count}."
                )

            validate_unique_key(
                persisted_df,
                keys,
                label
            )

            print(
                f"{label}: {persisted_count}/{expected_count} validated."
            )

        # ----------------------------------
        # MARK EXECUTION AS SUCCESS
        # ----------------------------------

        update_ingestion_log(
            execution_id=execution_id,
            execution_date=execution_date,
            source_status=source_status,
            ingestion_status="SUCCESS",
            started_at=started_at,
            response_count=response_count,
            item_count=item_count
        )

        successful_executions += 1

        print(f"Execution completed successfully: {execution_id}")

    except Exception as error:

        error_message = str(error)
        log_error_message = None

        try:
            update_ingestion_log(
                execution_id=execution_id,
                execution_date=execution_date,
                source_status=source_status,
                ingestion_status="FAILED",
                started_at=started_at,
                error_message=error_message
            )

        except Exception as log_error:
            log_error_message = str(log_error)

            print(
                f"CRITICAL: Failed to update ingestion log "
                f"for execution {execution_id}."
            )
            print(f"Original ingestion error: {error_message}")
            print(f"Ingestion log error: {log_error_message}")

        failed_executions.append({
            "execution_id": execution_id,
            "error": error_message,
            "log_error": log_error_message
        })

        print(f"Execution failed: {execution_id}")
        print(f"Original error: {error_message}")


In [0]:

# FINAL INGESTION SUMMARY
completed_count = len(completed_ids)
running_count = len(recent_running_ids)
stale_count = len(stale_running_ids)
failed_retry_count = len(failed_ids)
new_count = len(new_ids)

skipped_executions = completed_count + running_count

print("\n==========================================")
print("BRONZE INGESTION SUMMARY")
print("==========================================")
print(f"Discovered: {len(executions)}")
print(f"Processed successfully: {successful_executions}")
print(f"Failed in this run: {len(failed_executions)}")
print(f"Already completed: {completed_count}")
print(f"Currently running: {running_count}")
print(f"Stale RUNNING retried: {stale_count}")
print(f"Previously failed retried: {failed_retry_count}")
print(f"New executions: {new_count}")
print(f"Skipped: {skipped_executions}")

display(
    spark.table(TABLE_INGESTION_LOG)
    .orderBy(F.col("updated_at").desc())
)

if failed_executions:

    failed_ids_current_run = [
        item["execution_id"]
        for item in failed_executions
    ]

    raise RuntimeError(
        f"Bronze ingestion failed for {len(failed_ids_current_run)} execution(s): "
        f"{failed_ids_current_run}"
    )

print("All pending executions processed successfully.")
